Rota Maker

Times:
- Open Start = 8
- Close Finish = 11
- Busy Hours = 12-2, 5-7
- Quiet Hours = 3-4
- Shift Size = 4-8hrs

In [8]:
from dataclasses import dataclass, field
from datetime import date, time, datetime, timedelta

In [9]:
'''
class Employee:
  _id_counter = 1 #Class variable

  def __init__(self, name, contract_hours, max_shifts_per_week,
               max_hours_per_day, job_role):
    self.id = Employee._id_counter #Make a unique ID
    Employee._id_counter += 1 #Increase counter
    self.name = name #Who is it
    self.contract_hours = contract_hours #Min hours
    self.assigned_hours = 0 #How many hours are they working
    self.max_shifts_per_week = max_shifts_per_week #How many days are they in
    self.max_hours_per_day = max_hours_per_day #4hr or 12hr shift
    self.job_role = job_role #Supervisor or FOH Assistant
'''

@dataclass
class Employee:
  name: str
  contract_hours: float
  assigned_hours: float = 0.0

  @property
  def remaining(self) -> float:
    return self.contract_hours - self.assigned_hours

@dataclass(frozen=True)
class Shift:
  day: date
  start: time
  end: time
  hours: float


In [10]:
def make_week_shifts(start_day: date) -> list[Shift]:
  shifts = []
  for i in range(7):
    d = start_day + timedelta(days=i)
    shifts.append(Shift(d, time(8, 0), time(16, 0), 8.0))
    shifts.append(Shift(d, time(16, 0), time(23, 0), 7.0))
  return shifts

In [11]:
def build_rota(employees: list[Employee], shifts: list[Shift]):
  rota = []

  for shift in shifts:
    candidates = sorted(
        employees,
        key=lambda e: (e.remaining >= shift.hours, e.remaining),
        reverse=True
    )
    chosen = candidates[0]
    chosen.assigned_hours += shift.hours
    rota.append((shift, chosen.name))

  return rota


In [12]:
employees = [
    Employee("Aleks", 37.5),
    Employee("Lorna", 37.5),
    Employee("Ben", 21),
    Employee("V", 21),
    Employee("Kyle", 21),
]

shifts = make_week_shifts(date(2026, 3, 2))
rota = build_rota(employees, shifts)

for shift, name in rota[:6]:
  print(shift.day, shift.start, shift.end, shift.hours, "->", name)

print("\nTotals:")
for e in employees:
  print(e.name, e.assigned_hours, "/", e.contract_hours)

2026-03-02 08:00:00 16:00:00 8.0 -> Aleks
2026-03-02 16:00:00 23:00:00 7.0 -> Lorna
2026-03-03 08:00:00 16:00:00 8.0 -> Lorna
2026-03-03 16:00:00 23:00:00 7.0 -> Aleks
2026-03-04 08:00:00 16:00:00 8.0 -> Aleks
2026-03-04 16:00:00 23:00:00 7.0 -> Lorna

Totals:
Aleks 31.0 / 37.5
Lorna 29.0 / 37.5
Ben 16.0 / 21
V 14.0 / 21
Kyle 15.0 / 21
